In [ ]:
import os
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()

chroma_client = chromadb.HttpClient(host='localhost', port="8000")

colls = chroma_client.list_collections()
colls

In [3]:
# Load
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()

from langchain.vectorstores import Chroma
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)

len(all_splits)

all_splits[4]

docs = [item.page_content for item in all_splits]
metas = [item.metadata for item in all_splits]

In [ ]:
data

In [4]:
bgelarge_ef = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    model_name="BAAI/bge-base-en-v1.5"
)

In [5]:
chroma_vectorstore = Chroma(
    collection_name="my_collection", embedding_function=bgelarge_ef, client=chroma_client
)

retriever = chroma_vectorstore.as_retriever()



collection = chroma_client.get_or_create_collection(name="llama_collection", embedding_function=bgelarge_ef)
collection.count()
ids = list(range(len(all_splits)))

s_ids = [str(i) for i in ids]

In [6]:
collection.add(ids=s_ids, documents=docs, metadatas=metas)
collection.count()

130

In [7]:
results = collection.query(query_texts="What is an agent", n_results=4)

In [8]:
results

{'ids': [['75', '98', '118', '2']],
 'distances': [[0.7659913301467896,
   0.7761162519454956,
   0.8449885388707049,
   0.8492191433906555]],
 'embeddings': None,
 'metadatas': [[{'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:',
    'language': 'en',
    'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'title': "LLM Powered Autonomous Agents | Lil'Log"},
   {'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, 